# Auto MPG Regression Practice with GPT(web)

## 주제: 자동차 정보로 연비(MPG) 예측하기

이번 실습에서는 **Auto MPG 데이터셋**을 사용합니다.

Auto MPG 데이터셋은 자동차의 여러 속성을 이용해 자동차의 연비인 `mpg`를 예측하는 회귀 문제입니다.

---

## 오늘 실습의 목표

이번 노트북에서는 데이터셋을 불러오고, 데이터 구조를 확인하는 데까지만 진행합니다.

그 이후 과정은 여러분이 **GPT(web)**를 사용해서 직접 완성합니다.

---

## 여러분이 GPT(web)로 해결할 내용

1. 결측치 처리
2. 범주형 변수 처리
3. train/test split
4. feature scaling
5. PyTorch Tensor 변환
6. PyTorch 회귀 모델 정의
7. 모델 학습
8. MSE, MAE, R² 평가
9. 실제값과 예측값 시각화

---

## 회귀 문제 복습

Regression은 정해진 class를 고르는 문제가 아니라, **연속적인 숫자 값**을 예측하는 문제입니다.

이번 실습에서는 다음과 같은 문제를 풉니다.

```text
자동차의 정보 → 자동차 연비(mpg) 예측
```

예를 들어 자동차의 무게, 마력, 실린더 수, 생산연도 등을 보고 연비를 예측합니다.

## 1. 필요한 라이브러리 불러오기

이번 노트북에서는 데이터 로딩과 기본 확인까지만 진행합니다.

이후 PyTorch 모델링에 필요한 라이브러리는 여러분이 GPT(web)를 사용해 직접 추가해보세요.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas version:", pd.__version__)
print("seaborn version:", sns.__version__)

## 2. Auto MPG 데이터셋 불러오기

`seaborn.load_dataset("mpg")`를 사용하면 Auto MPG 데이터셋을 쉽게 불러올 수 있습니다.

Colab에서 별도의 파일 업로드 없이 실행할 수 있습니다.

In [ ]:
df = sns.load_dataset("mpg")

df.head()

## 3. 데이터 크기 확인하기

데이터가 몇 행, 몇 열로 구성되어 있는지 확인합니다.

In [ ]:
print("Data shape:", df.shape)

## 4. 컬럼 확인하기

각 컬럼이 어떤 정보를 담고 있는지 확인합니다.

주요 컬럼:

- `mpg`: 자동차 연비, 우리가 예측할 target
- `cylinders`: 실린더 수
- `displacement`: 배기량
- `horsepower`: 마력
- `weight`: 자동차 무게
- `acceleration`: 가속도
- `model_year`: 생산연도
- `origin`: 생산 지역
- `name`: 자동차 이름

In [ ]:
df.columns

## 5. 데이터 타입 확인하기

숫자형 컬럼과 문자형/범주형 컬럼을 구분해야 합니다.

PyTorch 모델은 숫자 Tensor를 입력으로 받기 때문에 문자형 데이터는 그대로 사용할 수 없습니다.

In [ ]:
df.info()

## 6. 기초 통계량 확인하기

숫자형 feature들의 범위를 확인합니다.

값의 범위가 서로 많이 다르면 scaling이 필요할 수 있습니다.

In [ ]:
df.describe()

## 7. 결측치 확인하기

결측치가 있으면 모델 학습 전에 처리해야 합니다.

예를 들어 `horsepower`에 결측치가 있을 수 있습니다.

In [ ]:
df.isnull().sum()

## 8. Target 변수 확인하기

이번 실습의 target은 `mpg`입니다.

즉, 모델이 예측해야 할 값은 자동차의 연비입니다.

In [ ]:
target_column = "mpg"

print("Target column:", target_column)
print(df[target_column].head())

## 9. Target 분포 시각화하기

`mpg` 값이 어떤 분포를 가지는지 확인합니다.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["mpg"], bins=20)
plt.xlabel("MPG")
plt.ylabel("Count")
plt.title("Distribution of MPG")
plt.grid(True)
plt.show()

## 10. Feature와 Target 관계 살펴보기

연비와 관련이 클 것 같은 변수들을 시각화해봅니다.

예를 들어 자동차 무게가 무거울수록 연비가 낮아질 가능성이 있습니다.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["weight"], df["mpg"], alpha=0.7)
plt.xlabel("Weight")
plt.ylabel("MPG")
plt.title("Weight vs MPG")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["horsepower"], df["mpg"], alpha=0.7)
plt.xlabel("Horsepower")
plt.ylabel("MPG")
plt.title("Horsepower vs MPG")
plt.grid(True)
plt.show()

## 11. 사용할 컬럼 생각해보기

모델 학습을 위해서는 입력 feature `X`와 정답 target `y`를 나누어야 합니다.

이번 실습에서는 다음과 같이 생각할 수 있습니다.

```text
y = df["mpg"]
X = df에서 "mpg"를 제외한 나머지 컬럼
```

하지만 `name` 컬럼은 자동차 이름이므로 초보자 실습에서는 제거하는 것이 좋습니다.

또한 `origin`은 범주형 변수이므로 one-hot encoding이 필요합니다.

In [ ]:
# 아직 전처리는 하지 않습니다.
# 아래 코드는 feature와 target을 어떻게 나눌지 생각해보기 위한 예시입니다.

y = df["mpg"]
X = df.drop(columns=["mpg"])

print("X columns:")
print(X.columns)

print("\ny shape:", y.shape)
print("X shape:", X.shape)

# 여기서부터는 GPT(web)를 사용해서 직접 해결하기

이제부터는 여러분이 GPT(web)를 사용해 회귀 실습을 완성합니다.

아래 프롬프트를 그대로 복사해서 GPT(web)에 물어보세요.

---

## GPT(web)에 입력할 프롬프트 1: 전체 실습 코드 요청

```text
저는 Colab에서 Auto MPG 데이터셋으로 PyTorch 회귀 실습을 하고 있습니다.

현재 데이터는 다음 코드로 불러왔습니다.

import seaborn as sns
df = sns.load_dataset("mpg")

target은 mpg이고, 자동차의 여러 feature를 사용해서 mpg를 예측하고 싶습니다.

초보자용으로 다음 과정을 포함한 PyTorch 회귀 코드를 작성해주세요.

1. 결측치 처리
2. name 컬럼 제거
3. origin 컬럼 one-hot encoding
4. X, y 분리
5. train/test split
6. StandardScaler로 X scaling
7. 필요하다면 y scaling
8. PyTorch Tensor 변환
9. DataLoader 생성
10. 간단한 MLP 회귀 모델 정의
11. MSELoss와 Adam optimizer 설정
12. 모델 학습
13. MSE, MAE, R²로 평가
14. 실제값 vs 예측값 scatter plot 시각화

각 코드마다 초보자가 이해할 수 있도록 주석을 달아주세요.
```

---

## GPT(web)에 입력할 프롬프트 2: 오류가 났을 때

```text
아래 코드를 Colab에서 실행했더니 오류가 났습니다.
오류 원인을 초보자도 이해할 수 있게 설명하고, 수정된 코드를 제시해주세요.

[여기에 오류 메시지와 코드를 붙여넣기]
```

---

## GPT(web)에 입력할 프롬프트 3: 성능 개선 요청

```text
Auto MPG 데이터셋으로 PyTorch 회귀 모델을 만들었습니다.
현재 R² score가 낮습니다.

성능을 개선하기 위해 다음 관점에서 조언해주세요.

1. 데이터 전처리
2. 모델 구조
3. learning rate
4. epoch 수
5. 과적합 방지
6. Linear Regression, RandomForest 같은 baseline과 비교

초보자 실습용으로 적절한 수준의 개선 방법을 알려주세요.
```

# GPT(web)가 준 답변을 검토할 때 체크할 것

GPT가 생성한 코드를 그대로 믿기보다, 아래 항목을 확인하세요.

## 1. Target이 올바른가?

이번 문제의 정답은 `mpg`입니다.

```python
y = df["mpg"]
```

## 2. `name` 컬럼을 제거했는가?

자동차 이름은 문자열이므로 초보자 실습에서는 제거하는 것이 좋습니다.

```python
df = df.drop(columns=["name"])
```

## 3. 결측치를 처리했는가?

`horsepower`에 결측치가 있을 수 있습니다.

```python
df = df.dropna()
```

또는 평균값으로 채울 수도 있습니다.

## 4. 범주형 변수 `origin`을 처리했는가?

PyTorch 모델은 문자열을 직접 입력받을 수 없습니다.

```python
df = pd.get_dummies(df, columns=["origin"], drop_first=True)
```

## 5. train/test split을 했는가?

모델이 학습한 데이터로만 평가하면 안 됩니다.

## 6. scaling을 했는가?

`weight`, `horsepower`, `displacement`처럼 값의 범위가 큰 feature가 있으므로 scaling이 필요합니다.

## 7. 마지막 출력 크기가 1인가?

회귀 문제에서는 예측값 하나를 출력해야 합니다.

```python
nn.Linear(hidden_dim, 1)
```

## 8. Loss function이 회귀용인가?

회귀 문제에서는 보통 다음을 사용합니다.

```python
nn.MSELoss()
```

`CrossEntropyLoss`는 classification용이므로 이번 문제에는 맞지 않습니다.

# 제출 과제

GPT(web)를 사용해서 아래 내용을 완성하세요.

## 제출 내용

1. GPT(web)에 입력한 프롬프트
2. GPT(web)가 생성한 코드
3. 직접 수정한 부분
4. 최종 MSE
5. 최종 MAE
6. 최종 R² score
7. 실제값 vs 예측값 그래프
8. 느낀 점

---

## 느낀 점 작성 예시

```text
이번 실습에서는 Auto MPG 데이터셋으로 자동차 연비를 예측했다.
결측치와 범주형 변수를 처리해야 PyTorch 모델에 데이터를 넣을 수 있다는 것을 알게 되었다.
또한 회귀 문제에서는 Accuracy가 아니라 MSE, MAE, R² 같은 지표를 사용한다는 점을 배웠다.
GPT(web)가 준 코드를 그대로 사용하지 않고, target과 loss function이 문제 유형에 맞는지 확인하는 것이 중요했다.
```

# 추가 도전 과제

시간이 남으면 아래 과제도 시도해보세요.

## 도전 1
PyTorch MLP 모델과 Linear Regression을 비교해보세요.

## 도전 2
PyTorch MLP 모델과 RandomForestRegressor를 비교해보세요.

## 도전 3
`weight`, `horsepower`, `displacement` 중 어떤 feature가 `mpg`와 가장 관련 있어 보이는지 시각화해보세요.

## 도전 4
GPT(web)에 다음 질문을 해보세요.

```text
Auto MPG 데이터셋에서 어떤 feature가 자동차 연비 예측에 가장 중요할까요?
Python 코드로 feature importance를 확인하는 방법을 알려주세요.
```